# Ingest dokumen ke Aurora RAG

Notebook ini membaca satu file teks UTF-8 dari komputer tempat kernel berjalan, lalu mengirim isinya ke endpoint `POST /ingest`. Isi disimpan ke collection `source_code`; `file_path` hanya metadata sumber, bukan path yang dibaca server.

## Sebelum mulai

1. Jalankan API Aurora dan Ollama.
2. Atur `AURORA_RAG_TOKEN` pada proses API; token akan diminta secara tersembunyi di notebook.
3. Pastikan model embedding tersedia di Ollama (default: `nomic-embed-text`). Untuk menjalankan `/query`, model chat juga harus tersedia (default: `llama3`).
4. Kernel notebook harus bisa mengakses API lewat URL yang Anda masukkan.

Notebook ini hanya mengirim teks; PDF/DOCX tidak diekstrak otomatis. Setiap request menambahkan satu dokumen tanpa chunking atau deduplikasi. Menjalankan cell ingest berulang akan menambahkan duplikat; API ini tidak menyediakan penghapusan satu dokumen.

In [ ]:
import getpass
import json
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

In [ ]:
API_BASE_URL = input("Aurora API base URL [http://127.0.0.1:8000]:").strip() or "http://127.0.0.1:8000"
API_BASE_URL = API_BASE_URL.rstrip("/")
RAG_TOKEN = getpass.getpass("AURORA_RAG_TOKEN:").strip()
if not RAG_TOKEN:
    raise ValueError("AURORA_RAG_TOKEN wajib diisi")

print(f"API: {API_BASE_URL}")
print("Token diterima; nilainya tidak ditampilkan.")

In [ ]:
def post_json(endpoint: str, payload: dict, timeout: int = 120) -> dict:
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = Request(
        f"{API_BASE_URL}{endpoint}",
        data=body,
        headers={
            "Authorization": f"Bearer {RAG_TOKEN}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Aurora API HTTP {exc.code}: {detail}") from exc

## Pilih file teks

Masukkan path file yang dapat dibaca kernel notebook, misalnya `./docs/guide.md`. Untuk kernel remote, file harus sudah tersedia pada mesin remote tersebut. File dibaca sebagai UTF-8.

In [ ]:
LOCAL_FILE = Path(input("Path file teks UTF-8:").strip()).expanduser()
if not LOCAL_FILE.is_file():
    raise FileNotFoundError(LOCAL_FILE)

DOCUMENT_TEXT = LOCAL_FILE.read_text(encoding="utf-8")
default_source = LOCAL_FILE.name
FILE_PATH = input(f"Metadata file_path [{default_source}]:").strip() or default_source
default_language = LOCAL_FILE.suffix.lstrip(".") or "text"
LANGUAGE = input(f"Metadata language [{default_language}]:").strip() or default_language

print(f"File: {LOCAL_FILE}")
print(f"Metadata file_path: {FILE_PATH}")
print(f"Language: {LANGUAGE}")
print(f"Ukuran teks: {len(DOCUMENT_TEXT):,} karakter")

## Kirim ke Chroma

Tinjau sumber dan metadata di atas. Ketik `INGEST` untuk mengirim; selain itu notebook membatalkan request. Jangan jalankan ulang cell ini kecuali memang ingin menambahkan salinan dokumen baru.

In [ ]:
confirmation = input("Ketik INGEST untuk mengirim dokumen:").strip()
if confirmation == "INGEST":
    ingest_result = post_json(
        "/ingest",
        {
            "code_content": DOCUMENT_TEXT,
            "file_path": FILE_PATH,
            "language": LANGUAGE,
        },
    )
    print(json.dumps(ingest_result, ensure_ascii=False, indent=2))
else:
    print("Dibatalkan; tidak ada request yang dikirim.")

## Uji retrieval (opsional)

Cell ini mengirim pertanyaan ke `POST /query`, yang mencari collection `source_code` dan mengembalikan jawaban model. Ia tidak menampilkan daftar snippet sumber. Pertanyaan kosong akan dilewati.

In [ ]:
QUESTION = input("Pertanyaan untuk RAG (kosongkan untuk melewati):").strip()
if QUESTION:
    query_result = post_json("/query", {"question": QUESTION})
    print(json.dumps(query_result, ensure_ascii=False, indent=2))
else:
    print("Uji retrieval dilewati.")